# CSF1R Gene Variants and their Association with Parkinson's Disease

- Date Created: 7/10/25
- Date Last Modified: 5/30/26
- Data: GP2 Release 10 IMP
- Description: This notebook runs case/control association analysis, and SKAT/O tests for PD vs Controls on CSF1R coding variants. Data is imputed genotype array (neurobooster)

## Summary

In [1]:
! wb resource mount

Successfully mounted workspace bucket resources.


## Imports

In [3]:
## Import the necessary packages 
import os
import numpy as np
import pandas as pd
import math
import sys
import subprocess
import statsmodels.api as sm
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import pathlib

## Package Installs

### PLINK

In [409]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink1.9 is already installed in /home/jupyter/tools/


In [404]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink2; then
echo "Plink2 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink2 \n    -------"
wget -N https://s3.amazonaws.com/plink2-assets/alpha5/plink2_linux_x86_64_20240820.zip
unzip -o plink2_linux_x86_64_20240820.zip
echo -e "\n plink2 downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink2 is already installed in /home/jupyter/tools/


### ANNOVAR

In [245]:
%%bash

# Install ANNOVAR:
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then

echo "annovar is already installed in /home/jupyter/tools/"
else
echo "annovar is not installed"
cd /home/jupyter/tools/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

annovar is already installed in /home/jupyter/tools/


In [246]:
%%capture
%%bash

# Install ANNOVAR: Download resources for annotation

cd /home/jupyter/tools/annovar/

perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20140902 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp30a humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

### RVTests

In [247]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/tools/rvtests
cd /home/jupyter/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

rvtests is already installed


In [248]:
# chmod to make sure you have permission to run the program
! chmod u+x /home/jupyter/tools/plink
! chmod u+x /home/jupyter/tools/plink2
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

In [596]:
# Set up working directory
ancestry = "SAS"
CHROM = "chr5"
GENE = "CSF1R"
WORK_DIR =  f"/home/jupyter/CSF1R_results/{ancestry}_R10_IMP"
VEP_DIR = f"/home/jupyter/vep_data"
DATA_DIR = f"/home/jupyter/workspace"

! mkdir -p {WORK_DIR}
%cd {WORK_DIR}

/home/jupyter/CSF1R_results/SAS_R10_IMP


## Covariate File

In [ ]:
# Let's load the master key
key = pd.read_csv(f'{DATA_DIR}/master_key_release10_final_vwb.csv', header=0)
print(key.shape)
key = key[['GP2ID','baseline_GP2_phenotype', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset',  "age_at_diagnosis",'race_for_qc','nba', 'wgs', 'nba_label', 'study_type']]
key.rename(columns = {'GP2ID': 'IID',
                      'baseline_GP2_phenotype':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'race_for_qc':'RACE',
                                     'age_of_onset':'AAO',
                                 "age_at_diagnosis":"AAD"}, inplace = True)

## Subset to keep ancestry of interest 
ancestry_key = key[key['nba_label']==ancestry].copy()
ancestry_key.reset_index(drop=True)

In [ ]:
# drop monogenic cases
monogenic_index = ancestry_key[ancestry_key["study_type"] == "Monogenic"].index

ancestry_key.drop(monogenic_index, inplace=True)
ancestry_key[["phenotype", "study_type"]].value_counts()

In [ ]:
# Load information about related individuals 
try:
    related_df = pd.read_csv(f'{DATA_DIR}/{ancestry}_release10_vwb.related')
    print(related_df.shape)
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])

    # remove related individuals
    print(f"Removing {len(related_list)} related individuals.")

    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]

    # Check size
    print(ancestry_key.shape)
    ancestry_key
except:
    print("Warning. No related samples file found. Removing 0 samples.")


(102, 9)
Removing 102 related individuals.
(691, 12)


In [600]:
# Convert phenotype to binary (1/2)

pheno_mapping = {"PD": 2, "Control": 1}
ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')
# Check value counts of pheno
ancestry_key['PHENO'].value_counts(dropna=False)

PHENO
2       328
1       270
<NA>     93
Name: count, dtype: Int64

In [ ]:
## Get the PCs
pcs = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release10/raw_genotypes/{ancestry}/{ancestry}_release10_vwb.eigenvec', sep='\t')
selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)


# Reset the index to remove any potential issues
pcs = pcs.reset_index(drop=True)


In [602]:
# Convert phenotype to binary (1/2)
## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
# Female = 2; Male = 1
sex_mapping = {"Female": 2, "Male": 1}
ancestry_key["SEX"] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')

# Check value counts of SEX
ancestry_key["SEX"].value_counts(dropna=False)

SEX
1    440
2    251
Name: count, dtype: Int64

In [ ]:
print(f"Dropping {(~ancestry_key['nba']==1).sum()} samples that are not genotyped")
print(f"Dropping {(ancestry_key['wgs']==1).sum()} samples that have whole genome sequencing available")

# only include samples that aren't in wgs
ancestry_key = ancestry_key[(ancestry_key["nba"]==1) & (ancestry_key["wgs"]==0)]

Dropping 0 samples that are not genotyped
Dropping 124 samples that have whole genome sequencing available


In [604]:
## Make covariate file
df = pd.merge(ancestry_key, pcs, on='IID')

print(f"Dropping {ancestry_key.shape[0] - df.shape[0]} samples with no pcs")
## Drop lines with missing pheno
print(f"Dropping {(df['PHENO'].isna()).sum()} samples with missing pheno")

df = df[df['PHENO'].notna()]

df["PHENO"].value_counts(dropna=False)


Dropping 1 samples with no pcs
Dropping 33 samples with missing pheno


PHENO
2    258
1    248
Name: count, dtype: Int64

In [605]:
df.columns

Index(['IID', 'phenotype', 'baseline_GP2_phenotype_for_qc', 'SEX', 'AGE',
       'AAO', 'AAD', 'RACE', 'nba', 'wgs', 'nba_label', 'study_type', 'PHENO',
       'PC1', 'PC2', 'PC3', 'PC4', 'PC5'],
      dtype='object')

In [606]:
## Clean up and keep columns we need 
df["#FID"] = 0
final_df = df[['#FID','IID', 'SEX', 'AGE', 'PHENO', 'AAO', 'AAD', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()
final_df.groupby(['PHENO'])['SEX'].value_counts()

PHENO  SEX
1      1      166
       2       82
2      1      168
       2       90
Name: count, dtype: Int64

In [607]:
## Make file of sample IDs to keep 
samples_toKeep = final_df[['#FID', 'IID']].copy()

samples_toKeep.to_csv(f'{WORK_DIR}/{ancestry}.samplestoKeep', sep = '\t', index=False, header=None)
samples_toKeep.shape

(506, 2)

In [608]:
final_df[["#FID", "IID", "PHENO"]].to_csv(f'{WORK_DIR}/{ancestry}_pheno.txt', sep = '\t', index=False, na_rep="NA", header=True)

In [609]:
## Save your covariate file
final_df.to_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep = '\t', index=False, na_rep= "NA", header=True)

### Get cohort summary stats

In [611]:
print(f"""
Number of {ancestry} PD cases: {(final_df["PHENO"] == 2).sum()}
Number of {ancestry} Controls: {(final_df["PHENO"] == 1).sum()}

PD age range: {round(final_df[final_df["PHENO"] == 2]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 2]["AGE"].std(),2)}
Controls age range: {round(final_df[final_df["PHENO"] == 1]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 1]["AGE"].std(),2)}

Cases in AAO analysis: {((final_df["PHENO"] == 2) & (final_df["AAO"].notna())).sum()}
Cases missing AAO: {((final_df["PHENO"] == 2) & (final_df["AAO"].isna())).sum()}

nan_counts:\n{final_df.isna().sum()}
""")


Number of SAS PD cases: 258
Number of SAS Controls: 248

PD age range: 64.9 +/- 10.28
Controls age range: 55.91 +/- 15.87

Cases in AAO analysis: 210
Cases missing AAO: 48

nan_counts:
#FID       0
IID        0
SEX        0
AGE      117
PHENO      0
AAO      296
AAD      330
PC1        0
PC2        0
PC3        0
PC4        0
PC5        0
dtype: int64



### Subset PLINK ancestry file

In [612]:
PLINK_PREFIX = f"{CHROM}_{GENE}"

In [ ]:
## extract region of interest from file, flanked by 200kb both sides
! /home/jupyter/tools/plink2 \
--pfile {DATA_DIR}/gp2_tier2_eu_release10/imputed_genotypes/{ancestry}/{CHROM}_{ancestry}_release10_vwb \
--chr 5 \
--from-bp 149853295 \
--to-bp 150313365 \
--make-pgen erase-dosage \
--out {WORK_DIR}/{PLINK_PREFIX}


In [ ]:
# filter variants to just including those with mac1, and by samples to Keep to make main file for analyses
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PLINK_PREFIX} \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--mac 1 \
--make-pgen \
--out {WORK_DIR}/{CHROM}_{ancestry}

In [615]:
## generate individual pfile for variant annotation

# Get one sample 
samples = pd.read_csv(f"{WORK_DIR}/{ancestry}.samplestoKeep", sep = "\t", header=None)
sample_id_1 = samples.iloc[0,0]
sample_id_2 = samples.iloc[0,1]
sample_id = f"{sample_id_1} {sample_id_2}"

# convert to vcf for annotation
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{CHROM}_{ancestry} \
--recode vcf id-paste=iid \
--indv {sample_id} \
--silent \
--out {WORK_DIR}/{ancestry}_vep_input

In [616]:
### Bgzip and Tabix (zip and index the file)
! bgzip -k -f {WORK_DIR}/{ancestry}_vep_input.vcf
! tabix -f -p vcf {WORK_DIR}/{ancestry}_vep_input.vcf.gz
! tabix -f -p vcf {WORK_DIR}/{ancestry}_vep_input.vcf.gz

In [617]:
# move to input directory
! mv {WORK_DIR}/{ancestry}_vep_input.vcf {VEP_DIR}/input/{GENE}_R10_IMP/{ancestry}_vep_input.vcf

## Annotation using VEP
- *ITSN1* from NCBI gene, with gene region flanked by 200kb on both sides
- hg38 (chr21:33442501-34099861) 

In [ ]:
# run container to generate VEP output
vep_cmd = [
    "docker", "run",
    "-v", f"{os.environ['HOME']}/vep_data:/data",
    "ensemblorg/ensembl-vep",
    "vep",
    "--cache",
    "--offline",
    "--force_overwrite",
    "--input_file", f"input/{GENE}_R10_IMP/{ancestry}_vep_input.vcf",
    "--output_file", f"output/{GENE}/{ancestry}_R10_imp_vep_output.txt",
    "--pick_allele",
    "--refseq",
    "--hgvs", # get hgvs annotations 
    "--use_transcript_ref",
    "--assembly", "GRCh38",
    "--fasta", "vep/.vep/homo_sapiens/114_GRCh38/Homo_sapiens.GRCh38.dna.toplevel.fa.gz",
    "--force_overwrite"
]
result = subprocess.run(vep_cmd)
if result.returncode == 0:
    print("VEP annotation complete!")
else:
    print("VEP failed :(")

2025-09-03 03:11:34 - INFO: BAM-edited cache detected, enabling --use_transcript_ref; use --use_given_ref to override this
VEP annotation complete!


In [619]:
# Remove header from vep output, subset to important columns and move to work directory

! grep -v "##" {VEP_DIR}/output/{GENE}/{ancestry}_R10_imp_vep_output.txt | cut -f 1-7,14 > {WORK_DIR}/{ancestry}_vep_output.txt

In [620]:
# read in file from VEP filtered output
gene = pd.read_csv(f"{WORK_DIR}/{ancestry}_vep_output.txt", sep="\t")

In [621]:
gene.columns

Index(['#Uploaded_variation', 'Location', 'Allele', 'Gene', 'Feature',
       'Feature_type', 'Consequence', 'Extra'],
      dtype='object')

In [622]:
gene

,#Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,Extra
0,chr5:149853313:C:T,5:149853313,T,133522,NM_133263.4,Transcript,3_prime_UTR_variant,IMPACT=MODIFIER;STRAND=1;GIVEN_REF=C;USED_REF=...
1,chr5:149853547:C:A,5:149853547,A,133522,NM_133263.4,Transcript,3_prime_UTR_variant,IMPACT=MODIFIER;STRAND=1;GIVEN_REF=C;USED_REF=...
2,chr5:149853574:G:A,5:149853574,A,133522,NM_133263.4,Transcript,3_prime_UTR_variant,IMPACT=MODIFIER;STRAND=1;GIVEN_REF=G;USED_REF=...
3,chr5:149853623:C:T,5:149853623,T,133522,NM_133263.4,Transcript,3_prime_UTR_variant,IMPACT=MODIFIER;STRAND=1;GIVEN_REF=C;USED_REF=...
4,chr5:149853992:A:G,5:149853992,G,133522,NM_133263.4,Transcript,3_prime_UTR_variant,IMPACT=MODIFIER;STRAND=1;GIVEN_REF=A;USED_REF=...
...,...,...,...,...,...,...,...,...
3258,chr5:150312269:C:G,5:150312269,G,-,-,-,intergenic_variant,IMPACT=MODIFIER
3259,chr5:150312298:C:T,5:150312298,T,-,-,-,intergenic_variant,IMPACT=MODIFIER
3260,chr5:150313095:G:A,5:150313095,A,-,-,-,intergenic_variant,IMPACT=MODIFIER
3261,chr5:150313154:G:A,5:150313154,A,-,-,-,intergenic_variant,IMPACT=MODIFIER


In [623]:
# ref seq ID for ITSN1 is 6453, filter for it to keep only ITSN1 variants
gene_subset = gene[gene["Gene"] == "1436"].copy()

In [624]:
# edit dataframe to create variants to keep file that matches plink format
gene_subset["Location"] = gene_subset["Location"].str.replace("21:", "")
variation_df = gene_subset["#Uploaded_variation"].str.split(":", expand=True)
location_df = gene_subset["Location"].str.split("-", expand=True)

location_df.columns = ["Start", "End"]
location_df["End"] = location_df.apply(lambda row: row["Start"] if row["End"] is None else row["End"], axis=1)

variation_df.columns = ["Chr", "Start", "Ref", "Alt"]
variation_df.drop(columns=["Start"], inplace=True)

fixed_columns = pd.concat([variation_df, location_df], axis=1)
gene_df = pd.concat([fixed_columns,gene_subset], axis=1)
hgvs_annotations = gene_df["Extra"].str.split(":", expand=True).iloc[:,2]
gene_df["HGVS"] = hgvs_annotations
# drop unnecessary columns
gene_df.drop(columns=["Location", "Feature", "Feature_type", "Extra"], axis=1, inplace=True)

gene_df["Gene"] = "ITSN1"
gene_df.rename(columns={"#Uploaded_variation":"SNP"}, inplace=True)

In [625]:
# get value counts to determine how many total annotated
gene_df["Consequence"].value_counts()

Consequence
intron_variant                                320
upstream_gene_variant                          30
synonymous_variant                             12
3_prime_UTR_variant                             8
missense_variant                                6
5_prime_UTR_variant                             3
splice_region_variant,intron_variant            2
missense_variant,splice_region_variant          1
splice_donor_region_variant,intron_variant      1
Name: count, dtype: int64

In [626]:
# write value_counts output to file
gene_df["Consequence"].value_counts().to_csv(f"{WORK_DIR}/{ancestry}_{GENE}_all_vep_consequences.txt", sep="\t")

In [627]:
# filter consequences to only missense, splicing, frameshift and stopgain, and other high priority consequences
# Source: https://useast.ensembl.org/info/genome/variation/prediction/predicted_data.html 

vep_coding_variants = gene_df[gene_df["Consequence"].str.contains('^splice|^missense|^stop_gained|frame|^protein_altering|transcript_ablation|transcript_amplification', regex=True)].copy()
vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                              6
splice_region_variant,intron_variant          2
splice_donor_region_variant,intron_variant    1
missense_variant,splice_region_variant        1
Name: count, dtype: int64

In [628]:
# filter out any synonymous variants
final_vep_coding_variants = vep_coding_variants[~(vep_coding_variants["Consequence"].str.contains('synonymous_variant', regex=False))].copy()
final_vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                              6
splice_region_variant,intron_variant          2
splice_donor_region_variant,intron_variant    1
missense_variant,splice_region_variant        1
Name: count, dtype: int64

In [629]:
final_vep_coding_variants.head()

,Chr,Ref,Alt,Start,End,SNP,Allele,Gene,Consequence,HGVS
1352,chr5,C,T,5:150057348,5:150057348,chr5:150057348:C:T,T,ITSN1,missense_variant,p.Arg753Gln
1385,chr5,A,G,5:150061716,5:150061716,chr5:150061716:A:G,G,ITSN1,"splice_region_variant,intron_variant",None
1441,chr5,G,A,5:150068208,5:150068208,chr5:150068208:G:A,A,ITSN1,"splice_region_variant,intron_variant",None
1442,chr5,C,T,5:150068212,5:150068212,chr5:150068212:C:T,T,ITSN1,"splice_donor_region_variant,intron_variant",None
1443,chr5,C,T,5:150068280,5:150068280,chr5:150068280:C:T,T,ITSN1,missense_variant,p.Ala521Thr


In [630]:
# Save ids in PLINK format
final_vep_coding_variants["SNP"].to_csv(f"{WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt", sep="\t",  index=False, header=False)

In [631]:
# save consequences annotation file for later
final_vep_coding_variants.to_csv(f"{WORK_DIR}/{ancestry}_{GENE}_vep_coding_variant_annotations.txt",  sep="\t",  index=False)

## Burden Analyses using RVTests


In [632]:
PFILE_PREFIX = f"{CHROM}_{ancestry}"

In [633]:
# get hg38 refFlat file from ucsc
! wget -nc -O /home/jupyter/refFlat.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
! gunzip -q -f -k /home/jupyter/refFlat.txt.gz

File ‘/home/jupyter/refFlat.txt.gz’ already there; not retrieving.


In [634]:
# make a pheno file for plink input
! cut -f 1,2,5 {WORK_DIR}/{ancestry}_covariate_file.txt > {WORK_DIR}/{ancestry}_pheno.txt

In [635]:
# Convert the files from Plink 2.0 to Plink 1.9 format 
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--make-bed \
--pheno {ancestry}_pheno.txt \
--pheno-name PHENO \
--max-alleles 2 \
--out {WORK_DIR}/{PFILE_PREFIX}

PLINK v2.00a5.14LM 64-bit Intel (20 Aug 2024)  www.cog-genomics.org/plink/2.0/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS.log.
Options in effect:
  --make-bed
  --max-alleles 2
  --out /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --pfile /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --pheno SAS_pheno.txt
  --pheno-name PHENO

Start time: Wed Sep  3 03:11:43 2025
12984 MiB RAM detected, ~9940 available; reserving 6492 MiB for main workspace.
Using up to 2 compute threads.
506 samples (172 females, 334 males; 506 founders) loaded from
/home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS.psam.
3263 variants loaded from
/home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS.pvar.
1 binary phenotype loaded (258 cases, 248 controls).
3263 variants remaining after main filters.
Writing /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS.fam ... done.
Writing /home/jupyter/CSF1R_results/SAS_R10_IMP/ch

In [ ]:
## extract variants as a vcf
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{PFILE_PREFIX} \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--recode vcf-iid \
--out {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn

In [637]:
### Bgzip and Tabix (zip and index the file)
! bgzip -f {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf
! tabix -f -p vcf {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz

In [638]:
# because of errors with rvtests reading in phenotypes, create cov file that matches rvtest pheno file header
rv_df = pd.read_csv(f'{WORK_DIR}/{ancestry}_covariate_file.txt', sep="\t")
rv_df.columns = rv_df.columns.str.lower()
rv_df["fid"] = rv_df["#fid"]
rv_df["fatid"] = 0
rv_df["matid"] = 0
rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age", "pc1", "pc2", "pc3", "pc4", "pc5"]]
rv_df.to_csv(f'{WORK_DIR}/rvtests_covariate_file.txt', sep='\t', index=False, header=True)

In [ ]:
## RVtests with covariates 
! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
--out {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn \
--kernel skat,skato \
--inVcf {WORK_DIR}/{ancestry}_{GENE}.coding_nonsyn.vcf.gz \
--pheno {WORK_DIR}/rvtests_covariate_file.txt \
--pheno-name pheno \
--gene {GENE} \
--geneFile /home/jupyter/refFlat.txt \
--covar {WORK_DIR}/rvtests_covariate_file.txt \
--covar-name sex,pc1,pc2,pc3,pc4,pc5

# --out : Name of output 
# --burden cmc --kernel skato: tests to run 
# --inVcf : VCF file 
# --gene: gene name (if only looking at one or a few)
# --geneFile refFlat.txt
# --pheno :  covar file
# --mpheno : # column that has phenotype information
# --pheno-name : column name with phenotype in file
# --covar : covar file
# --freqUpper : optional, MAF cut-off
# --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
## 1=controls; 2=cases

In [641]:
## look at results 
! cat {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn.Skat.assoc

Gene	RANGE	N_INFORMATIVE	NumVar	NumPolyVar	Q	Pvalue	NumPerm	ActualPerm	Stat	NumGreater	NumEqual	PermPvalue
CSF1R	5:150053290-150092624,5:150053290-150113372,5:150053294-150113365,5:150053294-150086554,5:150053294-150113365,5:150053294-150113365,5:150053290-150086640	506	10	10	2119.16	0.473565	10000	1969	2119.16	1000	0	0.507872


In [642]:
! cat {WORK_DIR}/{ancestry}_{GENE}.burden.coding_nonsyn.SkatO.assoc

Gene	RANGE	N_INFORMATIVE	NumVar	NumPolyVar	Q	rho	Pvalue
CSF1R	5:150053290-150092624,5:150053290-150113372,5:150053294-150113365,5:150053294-150086554,5:150053294-150113365,5:150053294-150113365,5:150053290-150086640	506	10	10	1059.58	0	0.663032


In [ ]:
## check to make sure file was created and saved
! ls {WORK_DIR}

## Case/Control Frequencies

### Run PLINK2 GLM


In [644]:
# run logistic regression for pvals and odds ratios
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--adjust \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--ci 0.95 \
--covar {WORK_DIR}/{ancestry}_covariate_file.txt \
--covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
--covar-variance-standardize \
--glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err \
--out {WORK_DIR}/{ancestry}_{GENE}

PLINK v2.00a5.14LM 64-bit Intel (20 Aug 2024)  www.cog-genomics.org/plink/2.0/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.log.
Options in effect:
  --adjust
  --ci 0.95
  --covar /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_covariate_file.txt
  --covar-name SEX,PC1,PC2,PC3,PC4,PC5
  --covar-variance-standardize
  --extract /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.all_variants_toKeep.txt
  --glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err
  --keep /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS.samplestoKeep
  --out /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R
  --pfile /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --pheno /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_pheno.txt

Start time: Wed Sep  3 03:11:45 2025
12984 MiB RAM detected, ~9941 available; reserving 6492 MiB for main workspace.
Using up to 2 

### Run Basic Association Test (fisher and 1df chi-sq)

In [645]:
BFILE_PREFIX = f"{CHROM}_{ancestry}"

In [646]:
## Run association test using Fisher's exact test
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--mpheno 1 \
--assoc fisher \
--ci 0.95 \
--allow-no-sex \
--out {WORK_DIR}/{ancestry}_{GENE}

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.log.
Options in effect:
  --allow-no-sex
  --assoc fisher
  --bfile /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --ci 0.95
  --extract /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.all_variants_toKeep.txt
  --keep /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS.samplestoKeep
  --mpheno 1
  --out /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R
  --pheno /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_pheno.txt

12984 MB RAM detected; reserving 6492 MB for main workspace.
3263 variants loaded from .bim file.
506 people (334 males, 172 females) loaded from .fam.
506 phenotype values present after --pheno.
--extract: 10 variants remaining.
--keep: 506 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 506 founders and 0

### Get variant frequencies

In [647]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--allow-no-sex \
--freq \
--out {WORK_DIR}/{ancestry}_{GENE}

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --extract /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.all_variants_toKeep.txt
  --freq
  --keep /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS.samplestoKeep
  --out /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R

12984 MB RAM detected; reserving 6492 MB for main workspace.
3263 variants loaded from .bim file.
506 people (334 males, 172 females) loaded from .fam.
506 phenotype values loaded from .fam.
--extract: 10 variants remaining.
--keep: 506 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 506 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132

### Merge files together

In [648]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ancestry}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ancestry}.samplestoKeep \
--pheno {WORK_DIR}/{ancestry}_pheno.txt \
--pheno-name PHENO \
--allow-no-sex \
--recode A \
--out {WORK_DIR}/{ancestry}_{GENE}

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/CSF1R_results/SAS_R10_IMP/chr5_SAS
  --extract /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R.all_variants_toKeep.txt
  --keep /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS.samplestoKeep
  --out /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_CSF1R
  --pheno /home/jupyter/CSF1R_results/SAS_R10_IMP/SAS_pheno.txt
  --pheno-name PHENO
  --recode A

12984 MB RAM detected; reserving 6492 MB for main workspace.
3263 variants loaded from .bim file.
506 people (334 males, 172 females) loaded from .fam.
506 phenotype values present after --pheno.
--extract: 10 variants remaining.
--keep: 506 people remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 506 founders and 0 nonfoun

In [649]:
### merge output files

ASSOC_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.assoc.fisher"
RECODE_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.raw"
GLM_HYBRID_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.PHENO.glm.logistic.hybrid"
GLM_ADJUST_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.PHENO.glm.logistic.hybrid.adjusted"
FREQ_FILE = f"{WORK_DIR}/{ancestry}_{GENE}.frq"


log_hybrid = pd.read_csv(GLM_HYBRID_FILE, sep="\s+", usecols=["ID", "A1", "A1_FREQ", "A1_CT", "ALLELE_CT", "P", "OR", "LOG(OR)_SE", "L95", "U95", "FIRTH?", "ERRCODE"])
log_hybrid.rename(columns={"ID":"SNP"}, inplace=True)

assoc_adjusted = pd.read_csv(GLM_ADJUST_FILE,  sep="\s+", usecols=["ID", "BONF"])
assoc_adjusted.rename(columns={"ID":"SNP"}, inplace=True)

assoc = pd.read_csv(ASSOC_FILE, sep="\s+", usecols=["SNP", "P", "F_A", "F_U"])
assoc.rename(columns={"P": "P_fisher"}, inplace=True)


freq = pd.read_csv(FREQ_FILE, sep='\s+', usecols=['SNP', 'MAF'])
df = pd.merge(log_hybrid, assoc_adjusted, on="SNP", how="right")
df = pd.merge(df, freq, on="SNP", how="left")
#merge freq with df
freq_assoc = pd.merge(assoc, df, on="SNP", how="left")

#read in recode file
recode = pd.read_csv(RECODE_FILE, sep="\s+")

# Pre-filter the dataset
cases_data = recode[recode["PHENOTYPE"] == 2]
controls_data = recode[recode["PHENOTYPE"] == 1]
# Make a list from the column names
column_names = recode.columns.tolist()

# Drop the first 6 columns to keep the variants 
variants = column_names[6:]
results = []
for variant in variants:
    # For cases
    hom_cases = cases_data[cases_data[variant] == 2].shape[0]
    het_cases = cases_data[cases_data[variant] == 1].shape[0]
    total_cases = cases_data.shape[0]
    # For controls
    hom_controls = controls_data[controls_data[variant] == 2].shape[0]
    het_controls = controls_data[controls_data[variant] == 1].shape[0]
    total_controls = controls_data.shape[0]
    results.append({
        "Variant": variant,
        "Hom Cases": hom_cases,
        "Het Cases": het_cases,
        "Total Cases": total_cases,
        "Hom Controls": hom_controls,
        "Het Controls": het_controls,
        "Total Controls": total_controls,
    })

# Return results
df_results = pd.DataFrame(results)
df_results["SNP"] = df_results["Variant"].apply(lambda x: x.rsplit("_", 1)[0])
df_results = df_results.drop("Variant", axis=1)

# Merge with assoc results 
full_results = pd.merge(freq_assoc, df_results, on="SNP", how="left")


##  get r2 for imputation 
! grep -v "##" {WORK_DIR}/{CHROM}_{ancestry}.pvar > {WORK_DIR}/{CHROM}_{ancestry}_imputation_info.txt

variant_file = pd.read_csv(f"{WORK_DIR}/{CHROM}_{ancestry}_imputation_info.txt", sep ="\t")
R2 = variant_file["INFO"].str.extract("(R2=\d+(\.\d+)?)", expand=False)[0].str.strip()

variant_file["R2"] = R2.str.strip("R2=")
variant_file["SNP"] = variant_file["ID"]
# make df with variant R2
variant_df = variant_file[["SNP", "R2"]].copy()

# append R2 to the final results dataframe
full_results_with_r = pd.merge(full_results, variant_df, on="SNP", how="left")
full_results_with_r
# append variant annotation
annotations = pd.read_csv(f"{WORK_DIR}/{ancestry}_{GENE}_vep_coding_variant_annotations.txt", sep="\t", usecols=["Consequence", "SNP", "HGVS"])
full_results_annotated = pd.merge(full_results_with_r, annotations, on="SNP", how="left")
full_results_annotated.head()
#subset to only columns to keep

clean_full_results = full_results_annotated[["SNP", "Consequence", "HGVS", "A1", "A1_CT", "A1_FREQ", "FIRTH?",
                                             "P", "P_fisher", "BONF", "OR", "LOG(OR)_SE","L95", "U95",
                                  "F_A", "F_U", "MAF", "R2",
                                   "Hom Cases", "Het Cases", "Total Cases", 
                                   "Hom Controls","Het Controls", "Total Controls", "ERRCODE"
                                   ]].copy()



clean_full_results.insert(1, "Ancestry", ancestry)

clean_full_results.head()

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_U,MAF,R2,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE
0,chr5:150057348:C:T,SAS,missense_variant,p.Arg753Gln,T,2,0.001996,Y,0.266230,0.49960,...,0.000000,0.001996,0.791193,0,2,258,0,0,248,.
1,chr5:150061716:A:G,SAS,"splice_region_variant,intron_variant",NaN,G,4,0.003960,N,0.987434,1.00000,...,0.004049,0.003960,0.960275,0,2,258,0,2,248,.
2,chr5:150068208:G:A,SAS,"splice_region_variant,intron_variant",NaN,A,114,0.114688,N,0.051245,0.07295,...,0.133600,0.114700,0.974759,1,46,258,7,52,248,.
3,chr5:150068212:C:T,SAS,"splice_donor_region_variant,intron_variant",NaN,T,2,0.001976,Y,0.336547,0.49970,...,0.000000,0.001976,0.888346,0,2,258,0,0,248,.
4,chr5:150068280:C:T,SAS,missense_variant,p.Ala521Thr,T,1,0.000988,Y,0.425858,0.49010,...,0.002016,0.000988,1,0,0,258,0,1,248,.


In [650]:
# Look at significant SNPs, if any 
sig_freq = clean_full_results[clean_full_results['P']<0.05]
sig_snps = sig_freq['SNP'].tolist()
sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
sig_df_results

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_U,MAF,R2,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE


In [651]:
# save files to working directory
clean_full_results.to_csv(f'{WORK_DIR}/{ancestry}_{GENE}_GP2_R10_imp.fullVariantInformation.txt', sep="\t", index=False)
sig_df_results.to_csv(f'{WORK_DIR}/{ancestry}_{GENE}_GP2_R10_imp.SignificantVariantInformation.txt' , sep="\t", index=False)

### 